In [1]:
# ============================================
#  Imports and device setup
# ============================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from torchvision import transforms, models
from torchvision.datasets import OxfordIIITPet

import os



In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [3]:
IMG_SIZE = 224      # Good default for ResNet
BATCH_SIZE = 32
EPOCHS = 5          
VAL_RATIO = 0.2     # Split trainval into train/val

DATA_DIR = "./data_Oxford"  # Folder where data will be stored

In [ ]:
# ImageNet normalization (ResNet was trained this way)
imagenet_mean = [0.485, 0.456, 0.406] # RGB has 3 chanels
imagenet_std  = [0.229, 0.224, 0.225]

# Training: resize, augment, normalize
train_transform = transforms.Compose([
    #resize
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    # augmentation
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)), # crop and resize to 244*244
    # normalizer
    transforms.ToTensor(), #change type to pytorch tensor
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

# Validation/Test: only resize + normalize, no augmentation
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])


In [5]:
# ============================================
#  Load Oxford-IIIT Pet dataset
#    We'll use:
#      - split="trainval" for train+val
#      - split="test" for test (optional later)
#    target_types="category" => breed label (0..36)
# ============================================
full_trainval_dataset = OxfordIIITPet(
    root=DATA_DIR,
    split="trainval",
    target_types="category",
    download=True,
    transform=train_transform  
)

test_dataset = OxfordIIITPet(
    root=DATA_DIR,
    split="test",
    target_types="category",
    download=True,
    transform=eval_transform
)

num_classes = len(full_trainval_dataset.classes)  # should be 37
print("Number of classes:", num_classes)
print("Classes (breeds):", full_trainval_dataset.classes)


100%|██████████| 792M/792M [04:04<00:00, 3.23MB/s] 
100%|██████████| 19.2M/19.2M [00:09<00:00, 1.95MB/s]


Number of classes: 37
Classes (breeds): ['Abyssinian', 'American Bulldog', 'American Pit Bull Terrier', 'Basset Hound', 'Beagle', 'Bengal', 'Birman', 'Bombay', 'Boxer', 'British Shorthair', 'Chihuahua', 'Egyptian Mau', 'English Cocker Spaniel', 'English Setter', 'German Shorthaired', 'Great Pyrenees', 'Havanese', 'Japanese Chin', 'Keeshond', 'Leonberger', 'Maine Coon', 'Miniature Pinscher', 'Newfoundland', 'Persian', 'Pomeranian', 'Pug', 'Ragdoll', 'Russian Blue', 'Saint Bernard', 'Samoyed', 'Scottish Terrier', 'Shiba Inu', 'Siamese', 'Sphynx', 'Staffordshire Bull Terrier', 'Wheaten Terrier', 'Yorkshire Terrier']


In [6]:
# ============================================
# 5. Train/Validation split
# ============================================
full_size = len(full_trainval_dataset)
val_size = int(VAL_RATIO * full_size)
train_size = full_size - val_size

train_dataset, val_dataset = random_split(
    full_trainval_dataset,
    [train_size, val_size]
)

# IMPORTANT:
# - Train dataset uses train_transform (with augmentation)
# - Val dataset should use eval_transform (no augmentation)
# We need to override the transform on the subsets:
train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform = eval_transform

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")


Train size: 2944, Val size: 736, Test size: 3669


In [7]:
# ============================================
#  DataLoaders
# ============================================
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True if device.type == "cuda" else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True if device.type == "cuda" else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True if device.type == "cuda" else False
)


In [9]:
# ============================================
#  Load pretrained ResNet18 and modify head
# ============================================

model = models.resnet18(pretrained=True)

# Freeze all layers (backbone)
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer:
# Original: model.fc = Linear(512 -> 1000)
# We want: 512 -> num_classes (37)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)

# Only the new fc layer is trainable
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

# Print model summary-ish info
print(model)
num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
num_total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {num_trainable:,} / {num_total:,}")


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [10]:
# ============================================
#  Loss function & optimizer
# ============================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.fc.parameters(),   # only train classifier head
    lr=1e-4
)

# ============================================
#  Training loop (simple: train loss + val loss)
# ============================================
for epoch in range(EPOCHS):
    # ----- Training -----
    model.train()
    running_train_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)           # shape: [batch, num_classes]
        loss = criterion(outputs, labels) # CrossEntropyLoss

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * images.size(0)

    epoch_train_loss = running_train_loss / len(train_loader.dataset)

    # ----- Validation -----
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * images.size(0)

    epoch_val_loss = running_val_loss / len(val_loader.dataset)

    print(f"Epoch [{epoch+1}/{EPOCHS}] - "
          f"Train Loss: {epoch_train_loss:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f}")

print("Training finished ✅")

Epoch [1/5] - Train Loss: 3.4674 | Val Loss: 3.1554
Epoch [2/5] - Train Loss: 2.9769 | Val Loss: 2.7128
Epoch [3/5] - Train Loss: 2.5636 | Val Loss: 2.3423
Epoch [4/5] - Train Loss: 2.2408 | Val Loss: 2.0169
Epoch [5/5] - Train Loss: 1.9625 | Val Loss: 1.8074
Training finished ✅


In [11]:
# ============================================
#  Test set evaluation (simple)
# ============================================
model.eval()  # evaluation mode

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)           # [batch, num_classes]
        _, preds = torch.max(outputs, 1)  # predicted class index

        correct += (preds == labels).sum().item()
        total += labels.size(0)

test_accuracy = correct / total
print(f"Test Accuracy: {test_accuracy:.4f}")


Test Accuracy: 0.7005
